# Notebook 5: Simulating population dynamics
## 5.0) Setup of the population dynamics problem
Population dynamics asks: given individuals that are born, reproduce, interact and die, how does the population number and their spatial distribution change over time. Just like in thermodynamics, instead of looking at each individual member of the species, you look at the macroscopic variables.

In [2]:
from Functions_final import distance_cells      
from Functions_final import final_history        
from Functions_final import kappa_exact     

ImportError: cannot import name 'kappa_exact' from 'Functions_final' (/Users/ignacio/Desktop/IPT Problem - Mousetrap Dynamics/Revised notebooks (copies)/Functions_final.py)

## 5.1) How the problem will be described

So far every run is a single cascade: armed traps fire, become spent, and never come back. The cascade consumes its own resource and always dies, so the model can only answer two questions: does a cascade invade ($\rho\kappa > 1$), and how much does it fire in total ($F = 1 - e^{-\rho\kappa F}$).

This is a special case of a population model. A fired trap is an individual that reproduces by firing others, uses up a finite resource (armed traps), and then dies (becomes spent). Each state of the lattice has a population reading:

| State | Mousetrap | Population |
|---|---|---|
| $-1$ | inert cell | unsuitable habitat / immune individual |
| $0$ | armed | susceptible / empty patch that can be colonised |
| $1$ | firing | infected / occupied, reproducing patch |
| $-2$ | spent | recovered / dead / extinct patch |

The model parameters translate in the same way:
- $\kappa$ is the offspring per individual per generation.
- $\rho\kappa$ plays the role of $R_0$.
- $f_k$ and $p_{\text{long}}$ form the dispersal kernel.
- $F = 1 - e^{-\rho\kappa F}$ is the Kermack–McKendrick final-size equation of the SIR epidemic.

What the model lacks is **renewal**. In real populations, births, recolonisation or waning immunity replace what is used up. I add one parameter, $p_{\text{regen}}$: the probability per generation that a spent trap is re-armed ($-2 \to 0$). With it the lattice becomes a spatial SIRS model, and the questions change from *"how big is the cascade?"* to the standard questions of population dynamics:

1. **Invasion:** does a small seed grow? (threshold $\rho\kappa > 1$, unchanged)
2. **Long-term state:** does the population settle at an equilibrium, oscillate, or go extinct?
3. **Regulation:** at equilibrium the armed density should settle near $S^* \approx 1/\kappa$, the level at which each firing trap exactly replaces itself.
4. **Stochastic extinction:** on a finite lattice a population that mean-field theory says should persist can die out by chance.

Each question is answered twice: by the lattice simulation, and by a mean-field map that extends the Poisson argument of Notebook 3:

$$
\begin{aligned}
\text{new}_t &= s_t\left(1 - e^{-\kappa i_t}\right) \\
s_{t+1} &= s_t - \text{new}_t + p_{\text{regen}}\, r_t \\
i_{t+1} &= \text{new}_t \\
r_{t+1} &= r_t + i_t - p_{\text{regen}}\, r_t
\end{aligned}
$$

Here $s, i, r$ are the fractions of the lattice that are armed, firing and spent, with $s + i + r = \rho$. Setting $p_{\text{regen}} = 0$ recovers the original model, which is the first consistency check.

4. Notebook outline
Mapping section. Put in the table above, and note that the final-size equation is the SIR one.
Sanity check. Run p_regen=0, p_persist=0. The final R fraction should reproduce your Notebook 3 curve.
Herd-immunity plot. Use the existing simulator with x-axis v = 1 − ρ. Mark v_c = 1 − 1/κ.
Turn on renewal (p_regen=0.05, n_seed=5). Plot S, I, R against t, simulation over mean-field. You should see damped oscillations and S → 1/κ.
Persistence diagram. Run many replicas over a grid of (ρ, p_regen). Plot P(I>0 at t=300), the persistence probability, as a heatmap. Overlay the mean-field boundary ρκ/(1−p_persist) = 1.
Extinction statistics. Plot the distribution of extinction times against L. Larger lattices should persist longer, which is the demographic-noise effect.
Locality. Repeat step 5 in 1D or with a short kernel. Measure how far the persistence boundary shifts from mean-field. This is the population-dynamics version of the gap in your README.